# 02 · Calibrazione sulle partite open di Metrica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrancescoRomeo02/more-than-a-hack-2027-InkSight/blob/team-solution/solution/notebooks/02_calibrazione_metrica.ipynb)

I parametri del modello (soglie, pesi, costi di cambio stato) **non** si possono tarare sulla partita della challenge.
Li calibriamo su **altre partite**: i due *Sample Game* pubblici di Metrica Sports (stesso fornitore, tracking + eventi).

- Le etichette H/A/D di quelle partite sono ricavate **in automatico dagli eventi** (`metrica.labels_from_events`): nessuna etichettatura a mano.
- Per avvicinarle ai nostri dati togliamo la palla a blocchi lunghi (come nel tracking automatico).
- Obiettivo: lo **spellF1 ufficiale** (codice di `evaluation/possession_metrics.py`).

⚠️ Questo è il notebook pesante: ~3,6 s per configurazione per core. Eseguilo sul server.

In [ ]:
# --- Setup: funziona sia in locale sia su Google Colab ----------------------------
# Su Colab: monta Drive, clona la repo in /content/mtah e legge i dati da DRIVE_DIR/data.
# params.json e la cache stanno direttamente in DRIVE_DIR; il CSV della submission viene copiato li'.
import os, sys, time, shutil, subprocess
REPO_URL  = "https://github.com/FrancescoRomeo02/more-than-a-hack-2027-InkSight.git"
BRANCH    = "team-solution"
DRIVE_DIR = "/content/drive/MyDrive/mtah2027"        # <- cartella su Drive con dentro data/tactical/...
IN_COLAB  = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/mtah"
    if not os.path.exists(os.path.join(ROOT, "games.py")):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    os.chdir(ROOT)
    # Tutto cio' che deve passare da un notebook all'altro sta su Drive (ogni notebook ha il suo runtime):
    os.environ.setdefault("MTAH_DATA_DIR", os.path.join(DRIVE_DIR, "data"))           # dati della challenge
    os.environ.setdefault("MTAH_CACHE_DIR", os.path.join(DRIVE_DIR, "cache"))         # ATD parsato, Metrica, log calibrazione
    os.environ.setdefault("MTAH_PARAMS", os.path.join(DRIVE_DIR, "params.json"))      # scritto da 02, letto da 03
    # (le variabili vanno impostate PRIMA di importare `possession`)
    if not os.path.isdir(os.path.join(os.environ["MTAH_DATA_DIR"], "tactical")):
        print("ATTENZIONE: dati non trovati in", os.environ["MTAH_DATA_DIR"], "- carica la cartella data/ su Drive")
    p = os.environ["MTAH_PARAMS"]
    print("params.json su Drive:", time.ctime(os.path.getmtime(p)) if os.path.exists(p) else "NON ANCORA PRESENTE (esegui il notebook 02)")
else:
    ROOT = os.environ.get("MTAH_ROOT") or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if not os.path.exists(os.path.join(ROOT, "games.py")):      # notebook lanciato dalla root
        ROOT = os.getcwd()

def save_to_drive(path):
    """Su Colab copia un file in DRIVE_DIR, cosi' non si perde quando il runtime si resetta."""
    if IN_COLAB:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        shutil.copy(path, DRIVE_DIR)
        print("copiato su Drive:", os.path.join(DRIVE_DIR, os.path.basename(path)))

sys.path[:0] = [ROOT, os.path.join(ROOT, "solution"), os.path.join(ROOT, "evaluation")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", category=RuntimeWarning)
print("ROOT =", ROOT, "| Colab" if IN_COLAB else "| locale")

In [ ]:
from possession import metrica, calibrate, emissions, decode
metrica.download()
G = metrica.load_game(1)
lab = metrica.labels_from_events(G)
print(pd.Series(lab).value_counts())
s = lab[lab != "X"]; print("segmenti:", int((s[1:] != s[:-1]).sum()) + 1)

In [ ]:
# Sanity check visivo delle etichette derivate dagli eventi
codes = {"H": 0, "A": 1, "D": 2, "X": np.nan}
plt.figure(figsize=(12, 2)); plt.plot(np.arange(20000)/25, [codes[c] for c in lab[:20000]], lw=0.8)
plt.yticks([0, 1, 2], ["H", "A", "D"]); plt.xlabel("s"); plt.title("Sample Game 1, primi ~13 minuti"); plt.show()

In [ ]:
P = calibrate.prepare(seed=0)
print("parametri di default -> spellF1 (media, per partita):", calibrate.score(P, emissions.PARAMS, decode.DECODE))

## Random search (parallela, con ripresa)

Ogni configurazione finita viene scritta subito in `calibration.jsonl` nella cache (su Colab: su Drive).
Se il runtime si interrompe, basta rieseguire il notebook: riparte da dove era arrivato.
Per un nuovo giro dopo aver cambiato il codice delle feature/emissioni, cambia `LOG` (i vecchi punteggi non valgono piu').

In [ ]:
from possession import data
N_TRIALS = 240                 # totale (per aumentarlo basta alzarlo e rieseguire: fa solo quelle mancanti)
N_JOBS = os.cpu_count() or 1
SEED = 2026
LOG = os.path.join(data.CACHE_DIR, "calibration_v3.jsonl")
print("log:", LOG)

t = time.time()
results = calibrate.parallel_search(N_TRIALS, N_JOBS, SEED, log_path=LOG)
print(f"{len(results)} configurazioni nel log, {time.time()-t:.0f}s in questa sessione")
df = pd.DataFrame([{"spellF1": r["spell"], "g1": r["per_game"][0], "g2": r["per_game"][1], **r["decode"],
                    **{k: r["params"][k] for k in calibrate.SEARCH}} for r in results]).sort_values("spellF1", ascending=False)
df.head(10).round(3)

Scegliamo la migliore. Per evitare di premiare un colpo di fortuna, controlliamo che vada bene su **entrambe** le partite (colonne g1, g2).

In [ ]:
from possession import pipeline
best = calibrate.best_of(results)     # criterio robusto: la peggiore delle due partite
print("migliore (max-min):", round(best["spell"], 3), [round(v, 3) for v in best["per_game"]], best["decode"])
path = calibrate.write_params(best, pipeline.params_path(), len(results), SEED)
print("scritto", path)
# classificatore dei buchi della palla: allenato su TUTTE le partite Metrica, finisce accanto a params.json
from possession import gaps
gpath = gaps.save(calibrate.fit_gap_model(P), pipeline.gap_model_path(),
                  source="Metrica Sample Games 1-2, atd_like ball, labels from events")
print("scritto", gpath, "| k_gap scelto:", round(best["params"]["k_gap"], 3))        # su Colab e' gia' su Drive: il notebook 03 lo legge da li'
if not IN_COLAB:
    print("in locale: committa solution/params.json se vuoi che la submission sia riproducibile dalla repo")